In [2]:
import pandas as pd
import sklearn

In [6]:
sold = pd.read_csv("combined_sold.csv")
print(sold.shape)

/tmp/ipykernel_6557/4185666676.py:1: DtypeWarning: Columns (2,45,78,79,80,81) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv("combined_sold.csv")


(552253, 82)


In [ ]:
 #Drop columns with a null rate of 50% or more
null_threshold = 0.5
         
for name, df in [('sold', sold)]:
    null_rate = df.isnull().mean()
    cols_to_drop = null_rate[null_rate >= null_threshold].index.tolist()
    print(f"{name}: dropping {len(cols_to_drop)} columns with >= {null_threshold:.0%} nulls -> {cols_to_drop}")
    df.drop(columns=cols_to_drop, inplace=True)
print(f"sold shape: {sold.shape}")

sold: dropping 25 columns with >= 50% nulls -> ['WaterfrontYN', 'BasementYN', 'CoListOfficeName', 'CoListAgentFirstName', 'CoListAgentLastName', 'FireplacesTotal', 'AssociationFeeFrequency', 'AboveGradeFinishedArea', 'TaxAnnualAmount', 'ElementarySchool', 'BuilderName', 'SubdivisionName', 'BuyerAgencyCompensationType', 'BuyerAgencyCompensation', 'TaxYear', 'BuildingAreaTotal', 'ElementarySchoolDistrict', 'CoBuyerAgentFirstName', 'BelowGradeFinishedArea', 'BusinessType', 'CoveredSpaces', 'MiddleOrJuniorSchool', 'HighSchool', 'LotSizeDimensions', 'MiddleOrJuniorSchoolDistrict']
sold shape: (552253, 57)


In [ ]:
# Drop agent/office/board identifiers -- these are IDs, not property features.
# One-hot encoding them would add 190+ near-useless columns and risks leakage
# (a specific agent/office can proxy for a price range).
sold.drop(columns=['BuyerOfficeAOR', 'BuyerAgentAOR', 'ListAgentAOR'], inplace=True)

# StateOrProvince is redundant with the more granular CountyOrParish -- drop it.
sold.drop(columns=['StateOrProvince'], inplace=True)

# Levels is ordinal ("One"/"Two"/"ThreeOrMore") but stored as comma-joined combos
# (e.g. "One,Two,MultiSplit"). Convert to a numeric max-level plus a separate
# MultiSplit flag instead of one-hot encoding all 15 raw combos. Done here, before
# the missing-indicator/imputation cells below, so NumLevels' nulls get the same
# group-wise median treatment as the other numeric columns.
level_map = {'One': 1, 'Two': 2, 'ThreeOrMore': 3}
sold['Levels_MultiSplit'] = sold['Levels'].str.contains('MultiSplit', na=False).astype(int)
sold['NumLevels'] = sold['Levels'].apply(
    lambda v: max((level_map[p] for p in v.split(',') if p in level_map), default=None)
    if pd.notna(v) else None
)
sold.drop(columns=['Levels'], inplace=True)

print(f"sold shape after dropping ID columns and re-encoding Levels: {sold.shape}")


In [ ]:
# Missing-indicator flags, created BEFORE imputation (must reflect original nulls)
numeric_cols = sold.select_dtypes(include='number').columns
cols_with_nulls = [c for c in numeric_cols if sold[c].isnull().any()]

for col in cols_with_nulls:
    sold[f'{col}_missing'] = sold[col].isnull().astype(int)

print(f"Added {len(cols_with_nulls)} missing-indicator columns: {cols_with_nulls}")

In [ ]:
# Group-wise median imputation with a fallback hierarchy:
# PropertySubType+ZIP -> PropertySubType+City -> PropertySubType+County -> PropertySubType -> global
# Each level only fills what the previous (more specific) level couldn't resolve.
group_hierarchy = [
    ['PropertySubType', 'PostalCode'],
    ['PropertySubType', 'City'],
    ['PropertySubType', 'CountyOrParish'],
    ['PropertySubType'],
    [],
]

for col in cols_with_nulls:
    for group_keys in group_hierarchy:
        if group_keys:
            group_medians = sold.groupby(group_keys)[col].transform('median')
            sold[col] = sold[col].fillna(group_medians)
        else:
            sold[col] = sold[col].fillna(sold[col].median())

remaining_nulls = sold[cols_with_nulls].isnull().sum().sum()
print(f"Remaining nulls in imputed columns after fallback chain: {remaining_nulls}")

In [ ]:
# CountyOrParish is meaningful but too high-cardinality (63 values) for one-hot --
# target-encode it as mean ClosePrice per county instead of adding 63 dummy columns.
# Note: this uses ClosePrice across the full dataset (train+test), same as the
# group-wise median imputation above -- consistent with, but not a fix for, that leakage.
county_means = sold.groupby('CountyOrParish')['ClosePrice'].transform('mean')
sold['CountyOrParish_MeanPrice'] = county_means.fillna(sold['ClosePrice'].mean())
sold.drop(columns=['CountyOrParish'], inplace=True)

# Convert categorical (non-numeric) columns to numeric via one-hot encoding.
# Date-named columns and very high-cardinality columns need different treatment
# (date feature engineering / target-encoding, not one-hot), so they're skipped here.
categorical_cols = sold.select_dtypes(exclude='number').columns
date_cols = [c for c in categorical_cols if 'Date' in c]

cardinality = sold[categorical_cols].nunique(dropna=True)
onehot_cols = [c for c in categorical_cols if c not in date_cols and 2 <= cardinality[c] <= 100]
high_card_cols = [c for c in categorical_cols if c not in date_cols and c not in onehot_cols]

print(f"One-hot encoding {len(onehot_cols)} columns: {onehot_cols}")
print(f"Skipping {len(date_cols)} date columns (need date feature engineering, not one-hot): {date_cols}")
print(f"Skipping {len(high_card_cols)} high-cardinality/ID-like columns (>100 unique values): {high_card_cols}")

sold = pd.get_dummies(sold, columns=onehot_cols, dummy_na=False, dtype=int)
print(f"sold shape after one-hot encoding: {sold.shape}")


In [ ]:
# Test set = most recent month of CloseDate. Train set = X months immediately
# preceding it, where X is tuned below rather than fixed.
sold['CloseDate'] = pd.to_datetime(sold['CloseDate'])
sold['CloseMonth'] = sold['CloseDate'].dt.to_period('M')

monthly_counts = sold['CloseMonth'].value_counts().sort_index()
print("Rows per month:")
print(monthly_counts)

test_month = sold['CloseMonth'].max()
print(f"\nMost recent month (test set): {test_month}  ({monthly_counts[test_month]} rows)")
print("Note: months before 2023-05 have far fewer rows (partial data collection) -- "
      "keep this in mind if a large X reaches back that far.")
sold.to_csv("cleaned_sold.csv")
print(sold.len())

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Tune X via a single-fold walk-forward validation: hold out the month right
# before the test month as a validation proxy, so the true test month is never
# touched while choosing X. This is a quick probe model just to rank candidate
# X values -- not the final model.
validation_month = test_month - 1
feature_cols = [c for c in sold.select_dtypes(include='number').columns if c != 'ClosePrice']

val_df = sold[sold['CloseMonth'] == validation_month]

results = []
for X in range(1, 25):
    train_start = validation_month - X
    train_df = sold[(sold['CloseMonth'] >= train_start) & (sold['CloseMonth'] < validation_month)]
    if len(train_df) < len(feature_cols) * 2:
        continue
    model = LinearRegression()
    model.fit(train_df[feature_cols], train_df['ClosePrice'])
    preds = model.predict(val_df[feature_cols])
    mae = mean_absolute_error(val_df['ClosePrice'], preds)
    results.append((X, len(train_df), mae))

results_df = pd.DataFrame(results, columns=['X_months', 'train_rows', 'val_mae'])
print(results_df.sort_values('val_mae'))

best_X = int(results_df.loc[results_df['val_mae'].idxmin(), 'X_months'])
print(f"\nBest X by validation MAE: {best_X} months")

In [ ]:
# Final split: test = most recent month, train = best_X months immediately before it
train_start = test_month - best_X
train_mask = (sold['CloseMonth'] >= train_start) & (sold['CloseMonth'] < test_month)
test_mask = sold['CloseMonth'] == test_month

train_set = sold[train_mask].copy()
test_set = sold[test_mask].copy()

print(f"Train: {train_start} to {test_month - 1}  ({len(train_set)} rows)")
print(f"Test:  {test_month}  ({len(test_set)} rows)")